In [ ]:
import pandas as pd
import tqdm, os
import pickle
import warnings
warnings.simplefilter("ignore")
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Subset

import sys
sys.path.insert(0, "../../..")
from utils.scaling import fit_scalers, apply_scaler
from utils.acoustics_processing import get_mfccs
from utils.articulation_processing import get_aemri75_articulatory_features

In [ ]:
df = pd.read_csv("../dataset_construction/datasets/processed_aemri75.csv")
df["subject"] = df["art_f"].apply(lambda x: os.path.basename(x).split("_")[0])

In [ ]:
df

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import numpy as np
from utils.models import MDN
import copy

from utils.models import VideoAutoencoder, SequenceAutoencoder

device = "cpu"

art_model = VideoAutoencoder(
    input_hw=(84, 84),
    latent_dim=64,        # was 1024. With ~30k tokens that was memorisation
                          # territory, and Euclidean distances concentrate badly
                          # in 1024-d, which is bad for what you use z for.
    input_channels=1,
    output_activation=None,
    stn_mode="shared",
    stn_transform="rigid"

).to(device)

ckpt = torch.load(
    "../../encoder_training/ckpts/aemri/art_model_ckpt.pth",
    map_location=device,
)
art_model.load_state_dict(ckpt["model"])
art_model.eval()

ac_model = SequenceAutoencoder(
    feature_shape=13,
    latent_dim=64,
).to(device)
ac_model.load_state_dict(torch.load("../../encoder_training/ckpts/aemri75/ac_model_ckpt.pth", map_location=device))
ac_model.eval()

In [ ]:
df

In [ ]:
import cv2
with open("../../encoder_training/scalers/aemri75_video_stats.pkl", "rb") as f:
    _REC_STATS = pickle.load(f)

def recording_stats(path, n_sample=400):
    """
    Pixel mean and sd over the WHOLE recording, cached per file.
 
    Note cv2's CAP_PROP_POS_FRAMES seeking on compressed video snaps to the
    nearest keyframe on some codecs. That is harmless for statistics, but it is
    worth checking separately that _read_video_segment returns frames that really
    correspond to the requested times — a silent half-second offset would
    misalign every token against the audio.
    """
    if path in _REC_STATS:
        return _REC_STATS[path]
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        cap.release()
        return None
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return None
    idxs = np.unique(np.linspace(0, total - 1, min(n_sample, total)).astype(int))
    vals = []
    for i in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
        ok, fr = cap.read()
        if ok:
            vals.append(fr[:, :, 0].astype(np.float32))
    cap.release()
    if not vals:
        return None
    v = np.stack(vals)
    _REC_STATS[path] = (float(v.mean()), float(v.std()) + 1e-5)
    return _REC_STATS[path]

with open(f"../../encoder_training/scalers/aemri75_ac.pkl", "rb") as f:
    ac_scalers = pickle.load(f)

In [ ]:
def get_vectors(art_f, ac_f, start, end, subject):
    stats = recording_stats(art_f)
    articulatory_features = get_aemri75_articulatory_features(
        art_f,
        start,
        end,
        stats=stats
    )

    if articulatory_features is not None:

        articulatory_features = (
            torch.from_numpy(articulatory_features)
            .float()
            .unsqueeze(0)
            .unsqueeze(0)
            .to(device)
        )

        art_out, art_z = art_model(
            articulatory_features,
            return_latent=True
        )
    else:
        art_out, art_z = None, None

    mfccs = get_mfccs(ac_f, start, end)

    if mfccs is not None:
        mfccs = apply_scaler(mfccs, ac_scalers[subject])
        mfccs = (
            torch.from_numpy(mfccs)
            .float()
            .unsqueeze(0)
            .to(device)
        )

        ac_out, ac_z = ac_model(mfccs, return_latent=True)
    else:
        ac_out, ac_z = None, None

    return {
        "articulatory_features": (
            articulatory_features.detach().cpu().numpy()
            if articulatory_features is not None
            else None
        ),
        "art_z": (
            art_z.detach().cpu().numpy()
            if art_z is not None
            else None
        ),
        "mfccs": (
            mfccs.detach().cpu().numpy()
            if mfccs is not None
            else None
        ),
        "ac_z": (
            ac_z.detach().cpu().numpy()
            if ac_z is not None
            else None
        ),
    }

In [ ]:
def distance(v1, v2):
    if v1 is None or v2 is None:
        return None
    return np.linalg.norm(v1 - v2)

In [ ]:
pbar = tqdm.tqdm(df.iterrows(), total=len(df))

dists = {
    "art_latent_dist": [],
    # "art_raw_dist": [],
    "ac_latent_dist": [],
    # "ac_raw_dist": [],
}
for idx, data in pbar:
    
    phone1_vectors = get_vectors(data["art_f"], data["ac_f"], data["phone1_start"], data["phone1_end"], data["subject"])
    phone2_vectors = get_vectors(data["art_f"], data["ac_f"], data["phone2_start"], data["phone2_end"], data["subject"])

    art_latent_dist = distance(phone1_vectors["art_z"], phone2_vectors["art_z"])
    # art_raw_dist = distance(phone1_vectors["articulatory_features"], phone2_vectors["articulatory_features"])

    ac_latent_dist = distance(phone1_vectors["ac_z"], phone2_vectors["ac_z"])
    # ac_raw_dist = distance(phone1_vectors["mfccs"], phone2_vectors["mfccs"])

    dists["art_latent_dist"].append(art_latent_dist)
    # dists["art_raw_dist"].append(art_raw_dist)
    dists["ac_latent_dist"].append(ac_latent_dist)
    # dists["ac_raw_dist"].append(ac_raw_dist)

pd.DataFrame(dists).to_csv("distances_aemri75.csv", index=False)